# Machine Learning Pipeline Introduction

This tutorial demonstrates the QuantStrata ML framework with a complete pipeline:

1. **Data Preparation** — Generate training data using MC simulation
2. **Model Definition** — Build a simple neural network
3. **Training** — Train the model using the generic training loop
4. **Evaluation** — Evaluate model performance with standardised metrics
5. **Inference** — Save, load, and run predictions

We'll train a simple MLP to approximate Black-Scholes option prices — a classic ML-for-pricing example.

---

## Prerequisites

```bash
pip install numpy matplotlib
```

No TensorFlow required — this example uses pure NumPy for simplicity.

In [ ]:
import sys
from pathlib import Path

# Add project root to path if needed
project_root = Path.cwd().parents[2]  # Adjust based on notebook location
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import matplotlib.pyplot as plt

# Set random seed for reproducibility
np.random.seed(42)

print("Imports successful!")

---

## 1. Data Preparation

We'll use the `build_pricing_dataset_from_mc` function to generate training data.
This simulates random option parameters and computes prices via Monte Carlo.

**Features:** spot, strike, vol, rate, expiry, option_type  
**Target:** option price

In [ ]:
from src.m_learning.data import build_pricing_dataset_from_mc, MLDataset

# Generate training data using Monte Carlo
# This samples random option parameters and computes prices via MC simulation
train_dataset = build_pricing_dataset_from_mc(
    n_samples=2000,       # Number of training samples
    n_paths=5000,         # MC paths per sample (more = less noise)
    spot_range=(80, 120), # Spot price range
    strike_range=(80, 120),
    vol_range=(0.1, 0.4),
    rate_range=(0.01, 0.08),
    expiry_range=(0.1, 1.5),
    seed=42,
)

print(f"Dataset shape: {train_dataset.features.shape}")
print(f"Target shape: {train_dataset.targets.shape}")
print(f"Feature names: {train_dataset.feature_names}")
print(f"Method: {train_dataset.metadata['method']}")

In [ ]:
# Split into train and test sets
train_data, test_data = train_dataset.split(train_ratio=0.8, seed=42)

print(f"Training samples: {len(train_data)}")
print(f"Test samples: {len(test_data)}")

# Preview the data
print("\nSample features (first 3 rows):")
print(train_data.features[:3])
print("\nSample targets (first 3 prices):")
print(train_data.targets[:3])

In [ ]:
# Normalise features for better training
# Store normalisation parameters for inference later

def normalise_features(features, mean=None, std=None):
    """Normalise features to zero mean and unit variance."""
    if mean is None:
        mean = features.mean(axis=0)
    if std is None:
        std = features.std(axis=0) + 1e-8  # Avoid division by zero
    return (features - mean) / std, mean, std

X_train_norm, feat_mean, feat_std = normalise_features(train_data.features)
X_test_norm, _, _ = normalise_features(test_data.features, feat_mean, feat_std)

# Also normalise targets (prices)
y_train = train_data.targets
y_test = test_data.targets
price_mean, price_std = y_train.mean(), y_train.std() + 1e-8
y_train_norm = (y_train - price_mean) / price_std
y_test_norm = (y_test - price_mean) / price_std

print(f"Feature mean: {feat_mean}")
print(f"Price mean: {price_mean:.2f}, std: {price_std:.2f}")

---

## 2. Model Definition

We'll build a simple 2-layer MLP (Multi-Layer Perceptron) that conforms to the `Trainable` protocol.

The protocol requires:
- `forward(inputs)` — Forward pass
- `compute_loss(y_true, y_pred)` — Loss computation
- `get_parameters()` — Return parameters for checkpointing
- `set_parameters(params)` — Load parameters
- `train_step(inputs, targets)` — (Optional) Single training step with gradient update

In [ ]:
class SimpleMLP:
    """
    Simple 2-layer MLP for option pricing.
    
    Architecture: input -> hidden (ReLU) -> hidden (ReLU) -> output
    
    Conforms to the Trainable protocol for use with the generic training loop.
    """
    
    def __init__(self, input_dim: int, hidden_dim: int = 32, output_dim: int = 1):
        """Initialise weights with Xavier initialization."""
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim
        
        # Layer 1: input -> hidden
        self.W1 = np.random.randn(input_dim, hidden_dim) * np.sqrt(2.0 / input_dim)
        self.b1 = np.zeros(hidden_dim)
        
        # Layer 2: hidden -> hidden
        self.W2 = np.random.randn(hidden_dim, hidden_dim) * np.sqrt(2.0 / hidden_dim)
        self.b2 = np.zeros(hidden_dim)
        
        # Layer 3: hidden -> output
        self.W3 = np.random.randn(hidden_dim, output_dim) * np.sqrt(2.0 / hidden_dim)
        self.b3 = np.zeros(output_dim)
        
        # Learning rate
        self.learning_rate = 0.01
        
        # Cache for backpropagation
        self._cache = {}
    
    def _relu(self, x):
        """ReLU activation."""
        return np.maximum(0, x)
    
    def _relu_grad(self, x):
        """ReLU gradient."""
        return (x > 0).astype(float)
    
    def forward(self, inputs: np.ndarray) -> np.ndarray:
        """Forward pass: inputs -> predictions."""
        # Layer 1
        z1 = inputs @ self.W1 + self.b1
        a1 = self._relu(z1)
        
        # Layer 2
        z2 = a1 @ self.W2 + self.b2
        a2 = self._relu(z2)
        
        # Output layer (linear)
        z3 = a2 @ self.W3 + self.b3
        
        # Cache for backprop
        self._cache = {'inputs': inputs, 'z1': z1, 'a1': a1, 'z2': z2, 'a2': a2, 'z3': z3}
        
        return z3.flatten() if z3.shape[1] == 1 else z3
    
    def compute_loss(self, y_true: np.ndarray, y_pred: np.ndarray) -> float:
        """Mean squared error loss."""
        return float(np.mean((y_true - y_pred) ** 2))
    
    def get_parameters(self) -> dict:
        """Return model parameters for checkpointing."""
        return {
            'W1': self.W1.copy(), 'b1': self.b1.copy(),
            'W2': self.W2.copy(), 'b2': self.b2.copy(),
            'W3': self.W3.copy(), 'b3': self.b3.copy(),
            'config': {'input_dim': self.input_dim, 'hidden_dim': self.hidden_dim},
        }
    
    def set_parameters(self, params: dict) -> None:
        """Load model parameters from checkpoint."""
        self.W1 = np.array(params['W1'])
        self.b1 = np.array(params['b1'])
        self.W2 = np.array(params['W2'])
        self.b2 = np.array(params['b2'])
        self.W3 = np.array(params['W3'])
        self.b3 = np.array(params['b3'])
    
    def train_step(self, inputs: np.ndarray, targets: np.ndarray) -> float:
        """
        Single training step: forward -> loss -> backward -> update.
        
        Returns the loss.
        """
        batch_size = len(inputs)
        
        # Forward pass
        y_pred = self.forward(inputs)
        loss = self.compute_loss(targets, y_pred)
        
        # Backward pass
        # dL/dz3 = 2 * (y_pred - y_true) / batch_size
        dz3 = 2 * (y_pred - targets).reshape(-1, 1) / batch_size
        
        # Layer 3 gradients
        dW3 = self._cache['a2'].T @ dz3
        db3 = dz3.sum(axis=0)
        da2 = dz3 @ self.W3.T
        
        # Layer 2 gradients
        dz2 = da2 * self._relu_grad(self._cache['z2'])
        dW2 = self._cache['a1'].T @ dz2
        db2 = dz2.sum(axis=0)
        da1 = dz2 @ self.W2.T
        
        # Layer 1 gradients
        dz1 = da1 * self._relu_grad(self._cache['z1'])
        dW1 = self._cache['inputs'].T @ dz1
        db1 = dz1.sum(axis=0)
        
        # Update weights (gradient descent)
        self.W3 -= self.learning_rate * dW3
        self.b3 -= self.learning_rate * db3
        self.W2 -= self.learning_rate * dW2
        self.b2 -= self.learning_rate * db2
        self.W1 -= self.learning_rate * dW1
        self.b1 -= self.learning_rate * db1
        
        return loss


# Create the model
model = SimpleMLP(input_dim=6, hidden_dim=32, output_dim=1)
print(f"Model created with {6 * 32 + 32 + 32 * 32 + 32 + 32 * 1 + 1} parameters")

---

## 3. Training

Now we'll use the generic training pipeline to train our model.

The `run_training` function handles:
- Batching
- Epoch iteration
- Validation tracking
- Checkpointing
- Early stopping

In [ ]:
from src.m_learning.core import TrainingConfig
from src.m_learning.pipeline import run_training
import tempfile

# Create a temporary directory for checkpoints
checkpoint_dir = tempfile.mkdtemp()

# Configure training
config = TrainingConfig(
    epochs=100,
    batch_size=64,
    learning_rate=0.01,  # Not used directly since model has its own
    validation_split=0.0,  # We'll use explicit validation data
    checkpoint_dir=checkpoint_dir,
    save_best_only=True,
    early_stopping_patience=10,  # Stop if no improvement for 10 epochs
    log_every=10,
    verbose=1,
)

print(f"Training config: {config.epochs} epochs, batch size {config.batch_size}")
print(f"Checkpoints will be saved to: {checkpoint_dir}")

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(message)s')

# Run training!
result = run_training(
    model=model,
    features=X_train_norm,
    targets=y_train_norm,
    config=config,
    val_features=X_test_norm,
    val_targets=y_test_norm,
)

print(f"\n=== Training Complete ===")
print(f"Final epoch: {result.final_epoch}")
print(f"Best epoch: {result.best_epoch}")
print(f"Best train loss: {result.best_train_loss:.6f}")
print(f"Best val loss: {result.best_val_loss:.6f}")
print(f"Training time: {result.training_time_seconds:.2f}s")

In [ ]:
# Plot training curves
fig, ax = plt.subplots(figsize=(10, 5))

epochs = range(1, len(result.history['loss']) + 1)
ax.plot(epochs, result.history['loss'], label='Training Loss', linewidth=2)
ax.plot(epochs, result.history['val_loss'], label='Validation Loss', linewidth=2)
ax.axvline(result.best_epoch, color='green', linestyle='--', label=f'Best Epoch ({result.best_epoch})')

ax.set_xlabel('Epoch')
ax.set_ylabel('Loss (MSE)')
ax.set_title('Training and Validation Loss')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_yscale('log')  # Log scale to see improvement better

plt.tight_layout()
plt.show()

---

## 4. Evaluation

Now let's evaluate the trained model using the standardised evaluation pipeline.

This computes:
- MSE, MAE, RMSE
- R² (coefficient of determination)
- Optional: pricing error vs benchmark

In [ ]:
from src.m_learning.pipeline import evaluate_model

# Evaluate on test set
eval_result = evaluate_model(
    model=model,
    features=X_test_norm,
    targets=y_test_norm,
    metrics=['mse', 'mae', 'rmse', 'r2'],
    training_result=result,  # Include training history
    metadata={'model': 'SimpleMLP', 'dataset': 'MC pricing'},
)

print("=== Evaluation Results (normalised scale) ===")
print(f"Loss: {eval_result.loss:.6f}")
print(f"MSE:  {eval_result.metrics['mse']:.6f}")
print(f"MAE:  {eval_result.metrics['mae']:.6f}")
print(f"RMSE: {eval_result.metrics['rmse']:.6f}")
print(f"R²:   {eval_result.metrics['r2']:.4f}")

In [ ]:
# Convert predictions back to original scale and evaluate
y_pred_norm = model.forward(X_test_norm)
y_pred_original = y_pred_norm * price_std + price_mean

# Compute metrics in original price scale
mae_original = np.mean(np.abs(y_test - y_pred_original))
rmse_original = np.sqrt(np.mean((y_test - y_pred_original) ** 2))
mape = np.mean(np.abs((y_test - y_pred_original) / (y_test + 1e-8))) * 100

print("\n=== Evaluation Results (original price scale) ===")
print(f"MAE:  ${mae_original:.2f}")
print(f"RMSE: ${rmse_original:.2f}")
print(f"MAPE: {mape:.2f}%")

In [ ]:
# Plot predictions vs actual
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot: predicted vs actual
ax1 = axes[0]
ax1.scatter(y_test, y_pred_original, alpha=0.5, s=10)
ax1.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2, label='Perfect prediction')
ax1.set_xlabel('Actual Price ($)')
ax1.set_ylabel('Predicted Price ($)')
ax1.set_title(f'Predicted vs Actual (R² = {eval_result.metrics["r2"]:.4f})')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Residual distribution
ax2 = axes[1]
residuals = y_test - y_pred_original
ax2.hist(residuals, bins=50, edgecolor='black', alpha=0.7)
ax2.axvline(0, color='red', linestyle='--', linewidth=2)
ax2.set_xlabel('Residual (Actual - Predicted) ($)')
ax2.set_ylabel('Frequency')
ax2.set_title(f'Residual Distribution (MAE = ${mae_original:.2f})')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## 5. Inference Pipeline

Finally, let's demonstrate the full inference pipeline:
1. Save the trained model
2. Load it back
3. Run predictions on new data

In [ ]:
from src.m_learning.pipeline import save_model, load_model, predict
import os

# Save the trained model
artifact_dir = tempfile.mkdtemp()
save_model(
    model=model,
    artifact_dir=artifact_dir,
    config=config,
    metadata={
        'model_type': 'SimpleMLP',
        'input_dim': 6,
        'hidden_dim': 32,
        'feat_mean': feat_mean.tolist(),
        'feat_std': feat_std.tolist(),
        'price_mean': float(price_mean),
        'price_std': float(price_std),
    }
)

print(f"Model saved to: {artifact_dir}")
print(f"\nArtifact contents:")
for f in os.listdir(artifact_dir):
    size = os.path.getsize(os.path.join(artifact_dir, f))
    print(f"  - {f} ({size} bytes)")

In [ ]:
# Load the model back
def model_factory():
    """Factory function to create a new model instance."""
    return SimpleMLP(input_dim=6, hidden_dim=32, output_dim=1)

loaded_model = load_model(artifact_dir, model_factory)
print("Model loaded successfully!")

# Verify it produces the same predictions
y_pred_loaded = predict(loaded_model, X_test_norm)
print(f"\nPrediction difference (should be ~0): {np.abs(y_pred_loaded - y_pred_norm).max():.2e}")

In [ ]:
# Demonstrate inference on new, unseen data
print("\n=== Inference on New Options ===")

# Create some sample options to price
new_options = np.array([
    # [spot, strike, vol, rate, expiry, option_type]
    [100, 100, 0.20, 0.05, 0.5, 1],   # ATM call, 6 months
    [100, 100, 0.20, 0.05, 0.5, -1],  # ATM put, 6 months
    [100, 110, 0.25, 0.05, 1.0, 1],   # OTM call, 1 year
    [100, 90, 0.25, 0.05, 1.0, -1],   # OTM put, 1 year
    [100, 100, 0.30, 0.03, 0.25, 1],  # ATM call, 3 months, high vol
])

# Normalise using stored parameters
new_options_norm = (new_options - feat_mean) / feat_std

# Predict
prices_norm = predict(loaded_model, new_options_norm)
prices = prices_norm * price_std + price_mean

# Display results
option_types = ['Call' if t == 1 else 'Put' for t in new_options[:, 5]]
print(f"{'Option':<15} {'Spot':>6} {'Strike':>7} {'Vol':>5} {'Expiry':>7} {'Price':>10}")
print("-" * 55)
for i, (opt, price) in enumerate(zip(new_options, prices)):
    print(f"{option_types[i]:<15} {opt[0]:>6.0f} {opt[1]:>7.0f} {opt[2]:>5.0%} {opt[3]:>7.2f} ${price:>9.2f}")

---

## 6. Save Evaluation Results

The evaluation results can be serialised to JSON for logging and comparison.

In [ ]:
# Save evaluation results to JSON
eval_path = os.path.join(artifact_dir, 'evaluation_results.json')
eval_result.to_json(eval_path)

print(f"Evaluation results saved to: {eval_path}")

# Also save training results
train_path = os.path.join(artifact_dir, 'training_results.json')
result.to_json(train_path)

print(f"Training results saved to: {train_path}")

# Show what was saved
print(f"\nFinal artifact directory contents:")
for f in sorted(os.listdir(artifact_dir)):
    size = os.path.getsize(os.path.join(artifact_dir, f))
    print(f"  - {f} ({size} bytes)")

---

## Summary

In this tutorial, we demonstrated the complete QuantStrata ML pipeline:

| Step | Component | What it does |
|------|-----------|-------------|
| **Data** | `build_pricing_dataset_from_mc()` | Generate training data via MC simulation |
| **Model** | `Trainable` protocol | Define model with forward, loss, get/set parameters |
| **Training** | `run_training()` | Generic training loop with batching, validation, checkpointing |
| **Evaluation** | `evaluate_model()` | Standardised metrics (MSE, MAE, R²) |
| **Inference** | `save_model()`, `load_model()`, `predict()` | Save, load, and deploy trained models |

### Key Takeaways

1. **Data Preparation** — The `MLDataset` class and adapters (MC, analytic, calibration) provide standardised feature/target formats.

2. **Model-Agnostic Training** — Any model conforming to `Trainable` can use `run_training()` — pure NumPy, Keras, or PyTorch.

3. **Standardised Evaluation** — `EvaluationResult` provides consistent metrics and JSON serialisation for logging.

4. **Artifact Convention** — Models are saved with `parameters.json`, `config.json`, and `metadata.json` for reproducibility.

### Next Steps

- Use `build_pricing_dataset_from_analytic()` with a BSM pricer for exact labels
- Try the `KerasTrainableAdapter` with a TensorFlow model
- Explore `build_calibration_dataset()` for calibration ML tasks
- See the GNN-RNN hybrid model in `src/m_learning/models/` for portfolio pricing

In [ ]:
# Clean up temporary directories
import shutil
shutil.rmtree(checkpoint_dir, ignore_errors=True)
# Note: we keep artifact_dir for inspection if desired

print("Tutorial complete!")